# Part 1: Project init (3 min)

Create a RocketRide app from nothing. RocketRide's app template writes a working React app that the RocketRide shell can load: its manifest, build config and a hello screen. You'll replace the screen in Part 3.

Every app's id is `<developer id>.<name>`. Your organization claims its developer id once, in the RocketRide extension: open the **App Builder**, go to the **Deploy** tab, and claim one (letters and underscores, for example `acme_labs`).

**Watch for:** `App id:` with your developer id, then the files the template wrote. The first run downloads RocketRide's app packages and runs `pnpm install`, so give it a minute or two.

In [ ]:
# Load the shared setup from specter-mutual.ipynb, then connect (safe to re-run)
%pip install nbformat
%run ./specter-mutual.ipynb
await connect()

In [ ]:
# Create the app from RocketRide's template (safe to re-run)
if not developer_id():
    # Claimed in the extension while this notebook was connected? Reconnect to pick it up.
    await client.disconnect()
    await client.connect()
if not developer_id():
    raise RuntimeError("Your organization has no developer id yet. In VS Code, open the RocketRide "
                       "App Builder, go to the Deploy tab and claim one (letters and underscores, "
                       "for example acme_labs). Then run this cell again.")

# An app made under a different developer id can't deploy here, so start that one over.
manifest = APP_ROOT / "package.json"
if manifest.exists() and json.loads(manifest.read_text(encoding="utf-8"))["appManifest"]["id"] != app_id():
    shutil.rmtree(APP_ROOT)
if not APP_ROOT.exists():
    WORKSPACE.mkdir(exist_ok=True)
    await client.deploy.create_app("specter-mutual", workspace_root=str(WORKSPACE), developer_id=developer_id(),
                                   display_name="Specter Mutual", install=True, on_progress=print)

print()
print("App id:", app_id())
for path in sorted(APP_ROOT.rglob("*")):
    if path.is_file() and "node_modules" not in path.parts:
        print("  ", path.relative_to(APP_ROOT).as_posix())

## Tell RocketRide what the app needs

The manifest in `package.json` describes the app to RocketRide. The portal calls a pipeline as the signed-in user, so it needs `authenticated`, and it ships the pipeline files that live next to the app, so it needs `include`.

**Watch for:** the updated `appManifest` printed below.

In [ ]:
# Write the app's manifest: sign-in required, and ship the pipelines folder with the app (safe to re-run)
manifest = APP_ROOT / "package.json"
package = json.loads(manifest.read_text(encoding="utf-8"))
DESCRIPTION = "A paranormal-insurance claims portal where an AI agent recommends and a policy engine decides. Built in RocketRide's Zero to Agent workshop."
package["description"] = DESCRIPTION
package["appManifest"].update(description=DESCRIPTION, authenticated=True, include=["pipelines"])
manifest.write_text(json.dumps(package, indent=2) + "\n", encoding="utf-8")
PIPELINES.mkdir(exist_ok=True)
print(json.dumps(package["appManifest"], indent=2))

In [ ]:
# Skip ahead: run this only if the stage above didn't work
# Starts the app over from a clean template, then writes the manifest.
await client.disconnect()
await client.connect()  # fresh identity, in case the developer id was just claimed
if not developer_id():
    raise RuntimeError("Still no developer id. Claim one in the App Builder's Deploy tab first.")
shutil.rmtree(APP_ROOT, ignore_errors=True)
WORKSPACE.mkdir(exist_ok=True)
await client.deploy.create_app("specter-mutual", workspace_root=str(WORKSPACE), developer_id=developer_id(),
                               display_name="Specter Mutual", install=True)
replay("01-project-init.ipynb")
print("Created a fresh app:", app_id(), "Carry on with Part 2: 02-pipeline-build.ipynb")

**Next:** [Part 2: Pipeline build](02-pipeline-build.ipynb)